A netwok has 3 components - an input layer, one or more hidden layers and one output layer. Information travels from left to right.

Input layer is not composed of neurons. It just contains raw data and number of nodes = number of features of the data.

Hidden layers are the brain of a neural network. Width of a hidden layer is number of neurons in it and depth is the number of layers. Each neuron in a hidden layer is connected to all outputs of the previous layer. They detect complex patterns like layer 1 may detect shapes, layer 2 may combine shapes to make objects, etc.

Output layer contains number of neurons = number of outputs we want. Activation function used in this is task specific.

# **FROM NEURON TO LAYER**

Say we have n neurons each having m weights. We can then make a weight matrix W of shape m x n where each column vector of the matrix corresponds to the weight vector of that neuron. Similarly we can have a bias vector of size n where each element represents bias of that neuron.

For a single neuron, linear step was wTx where w was weight vector. For this operation it is the matrix vector operation x @ W + b and result is a vector containing pre-activation of all neurons.

In [143]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Layer(nn.Module):
  def __init__(self, n_input, n_neurons, activation = None):
    super().__init__()
    self.weights = nn.Parameter(torch.randn(n_input, n_neurons)) #Use nn.Parameter(torch.randn()) instead of torch.randn() directly because nn.Parameter tells PyTorch it is a learnable parameter. Only registered parameters are updated by torch.optim.
    self.bias = nn.Parameter(torch.zeros(n_neurons)) #Without nn.Parameters, gradients can still be track upon setting requires_grad = True but we need to pass it in optim everytime.
    self.activation = activation

  def forward(self, x):
    logits = x @ self.weights + self.bias
    if self.activation is not None:
      return self.activation(logits)
    else:
      return logits

In [144]:
n_neurons = 3
n_inputs = 5
batch_size = 2

my_layer = Layer(n_inputs, n_neurons, activation = F.relu)
x = torch.randn(batch_size, n_inputs)
output = my_layer.forward(x)
print(output)

tensor([[0.0000, 0.5356, 0.0000],
        [0.0000, 1.2877, 1.4851]], grad_fn=<ReluBackward0>)


**Practice 1 -** Create an output layer for binary classification problem that should take 16 inputs from a previous hidden layer, using sigmoid as the activation function.

In [145]:
n_inputs = 16
n_neurons = 1 #because output is either yes or no. No other features.

my_layer = Layer(n_inputs, n_neurons, activation = F.sigmoid)
x = torch.randn(n_inputs)
output = my_layer.forward(x)
print(output)


tensor([0.9973], grad_fn=<SigmoidBackward0>)


**Practice 2 -** Create a Layer instance that would be suitable as the output layer for a regression problem (like predicting a house price). It should take 8 inputs.

In [146]:
n_inputs = 8
n_neurons = 1 #We are just interested in the price so one output - that single number.
batch_size = 5

my_layer = Layer(n_inputs, n_neurons) #No activation as the problem is generally Linear Regression. No need for activation as we don't need to deal with boundedness.

x = torch.randn(batch_size, n_inputs)

output = my_layer.forward(x)
print(output)

tensor([[-1.2073],
        [ 1.5324],
        [ 7.4535],
        [ 1.4551],
        [-0.8552]], grad_fn=<AddBackward0>)


**PyTorch built-in Layer Class -** nn.Linear does the same thing as our defined layer.

In [147]:
layer1 = nn.Linear(in_features=8, out_features=1)
x = torch.randn(batch_size, n_inputs)
output = layer1(x)
print(output)

tensor([[-0.0073],
        [-0.0977],
        [-1.6053],
        [-0.0679],
        [ 0.2932]], grad_fn=<AddmmBackward0>)


# **FROM LAYERS TO NETWORKS**

**Goal -** To implement a class for creating a neural network by stacking up layers together.

**To Implement -** A neural network with input layer (2 features), hidden layer 1 (16 neurons, ReLU Activation), hidden layer 2 (8 neurons, ReLU Activation), Output layer (1 neuron, Sigmoid Activation).

In [148]:
class SimpleNetwork(nn.Module):
  def __init__(self, n_inputs, n_hidden1, n_hidden2, n_outputs):
    super().__init__()
    self.hidden1 = Layer(n_inputs, n_hidden1, activation = F.relu)
    self.hidden2 = Layer(n_hidden1, n_hidden2, activation = F.relu)
    self.output = Layer(n_hidden2, n_outputs, activation = F.sigmoid)

    def forward(self, x):
      x = self.hidden1(x)
      x = self.hidden2(x)
      x = self.output(x)
      return x


In [149]:
network = SimpleNetwork(2, 16, 8, 1)
print(network)

SimpleNetwork(
  (hidden1): Layer()
  (hidden2): Layer()
  (output): Layer()
)


In [150]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16,8),
    nn.ReLU(),
    nn.Linear(8,1),
    nn.Sigmoid()
)

In [151]:
model

Sequential(
  (0): Linear(in_features=2, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
  (5): Sigmoid()
)

 **Practice 1 -** Implement a deeper network which has a third hidden layer of 4 neurons.

In [152]:
# 1. nn.Sequential implementation

model = nn.Sequential(
    nn.Linear(2,16),
    nn.ReLU(),
    nn.Linear(16,8),
    nn.ReLU(),
    nn.Linear(8,4),
    nn.ReLU(),
    nn.Linear(4,1),
    nn.Sigmoid()
)

print(model)

Sequential(
  (0): Linear(in_features=2, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=4, bias=True)
  (5): ReLU()
  (6): Linear(in_features=4, out_features=1, bias=True)
  (7): Sigmoid()
)


In [153]:
#2 Custom Implementation

class DeepNetwork(nn.Module):
  def __init__(self, input_size, hidden1_size, hidden2_size, hidden3_size, output_size):
    super().__init__()
    self.hidden1 = Layer(input_size, hidden1_size, activation = nn.ReLU)
    self.hidden2 = Layer(hidden1_size, hidden2_size, activation = nn.ReLU)
    self.hidden3 = Layer(hidden2_size, hidden3_size, activation = nn.ReLU)
    self.output = Layer(hidden3_size, output_size, activation = nn.Sigmoid)

  def forward(self, x):
    x = self.hidden1(x)
    x = self.hidden2(x)
    x = self.hidden3(x)
    x = self.output(x)
    return x


my_network = DeepNetwork(2,16,8,4,1)
print(my_network)

DeepNetwork(
  (hidden1): Layer()
  (hidden2): Layer()
  (hidden3): Layer()
  (output): Layer()
)


**Practice 2 -** Define a network for a regression task. It should take 10 input features, have one hidden layer of 32 neurons, and produce a single continuous output. What activation function should you use on the output layer?



In [154]:
class RegressionNetwork(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden = Layer(input_size, hidden_size, activation = nn.ReLU)
    self.output = Layer(hidden_size, output_size) #No activation on output since I don't want to bound the output.

  def forward(self, x):
    x = self.hidden(x)
    x = self.output(x)
    return x

In [155]:
task = RegressionNetwork(10, 32, 1)
print(task)

RegressionNetwork(
  (hidden): Layer()
  (output): Layer()
)


# **BACKPROPAGATION**

dL/dW -> How loss changes if we change the weights.


dL/db -> How loss changes if we change the bias.

For a single neuron we have -

z = w@x + b

a = activation(z)

So, using chain rule,

dL/dW = dL/da * da/dz * dz/dw

If dL/dw is positive, it means increasing w increases the loss. So we should decrease w.

If dL/dw is negative, increasing w decreases the loss. So we should increase w.

**Backward Pass**

Consider a 2 layer network -

Input - x

Hidden Layer -

  z1 = W1 @ x + b1

  a1 = f(z1)

Output -

  z2 = W2 @ a1 + b2

  a2 = f(z2)



Loss L is calculated based on found label a2 and true label y. Find W1, W2, b1, b2.



**Start from the end and move back!**

dL/dW2 = dL/da2 * da2/dz2 * dz2/dW2 = dL/da2 * f'(z2) * a1

Similarly, dL/db2 = dL/da2 * f'(z2) * 1

Same can be done for dL/dW1 and dL/db1

Say we use mean square error for our loss then L = (a2 - y)**2. So, dL/da2 = 2(a2 - y).



Using the results above -

dL/dW2 = 2*(a2 - y)*f'(z2)*x

dL/db2 = 2*(a2 - y)*f'(z2)

But for the hidden layer, finding dL/da1 is hard because a1 effects a2 via z2, not directly. But we can just use the chain rule again using this information.

dL/da1 = dL/z2 * dz2/da1 = dL/da2 * da2/dz2 * dz2/da1

= 2*(a2 - y)*f'(z2)*W2

Using the same analogy as for dL/dW2 and dL/db2 we can have -

dL/dW1 = dL/da1 * da1/dz1 * dz1/dW1 = 2 * (a2-y) * f'(z2) * W2 * f'(z1) * x

dL/db1 = dL/da1 * da1/dz1 * dz1/db1 = 2 * (a2-y) * f'(z2) * W2 * f'(z1)


The gradient flow starts at the loss function and flows backward through the network and hence is called backpropagation.

# **CODING BACKPROPAGATION**

For simplicity, use f(x) = x as an activation function so that f'(x) = 1.

In [156]:
import numpy as np

class CompleteLayer:
  def __init__(self, input_size, output_size):
    self.x = None
    self.weights = np.random.randn(input_size, output_size)
    self.bias = np.zeros((1, output_size))
    self.dL_dW = None
    self.dL_db = None
    self.dL_dI = None
    self.output = None

  def forward_pass(self, x):
    self.x = x
    return np.dot(self.x, self.weights) + self.bias

  def backward_pass(self, dL_dO): #d_output is dL/d(output of this layer)
    self.dL_dW = np.dot(self.x.T, dL_dO)
    self.dL_db = np.sum(dL_dO, axis=0, keepdims=True)
    self.dL_dI = np.dot( dL_dO, self.weights.T)
    return self.dL_dI #x of nth layer is output of last layer so this ensures we move back in each layer.

  def update(self, lr): #The update rule uses the core of gradient descent
    self.weights -= lr*self.dL_dW
    self.bias -= lr*self.dL_db

In [157]:
layer = CompleteLayer(3, 2)
input = np.array([[2, 1, 3], [1, 1, 1]])
layer.forward_pass(input)

array([[-1.8954165 , -1.467816  ],
       [-0.58225633, -0.53397818]])

In [158]:
class NeuralNetwork:
  def __init__(self):
    self.layers = []

  def add_layer(self, layer):
    self.layers.append(layer)

  def forward(self, x):
    for layer in self.layers:
      x = layer.forward_pass(x)
    return x

  def backward(self, dL_dO):
    for layer in reversed(self.layers):
      dL_dO = layer.backward_pass(dL_dO)

  def update(self, lr):
    for layer in self.layers:
      layer.update(lr)

In [159]:
nn = NeuralNetwork()
nn.add_layer(CompleteLayer(3, 2))
nn.add_layer(CompleteLayer(2, 1))

input_data = np.array([[0.2, 0.3, 0.1], [0.2, 0.4, 0.6]])
output = nn.forward(input_data)
print(output)

dL = np.array([[0.1], [0.3]])
nn.backward(dL)

[[0.10365688]
 [0.30981887]]


In [160]:
def mse(y, p):
  return np.mean((y - p)**2)

def d_mse(y, p):
  return 2*(y - p)/len(y)

In [161]:
nn = NeuralNetwork()

nn.add_layer(CompleteLayer(2,3))
nn.add_layer(CompleteLayer(3,1))

x = np.array([
    [0.1, 0.2],
    [0.3, 0.4],
    [0.5, 0.6],
    [0.7, 0.8]
])

y = np.array([
    [0.1],
    [0.2],
    [0.3],
    [0.4]
])

In [175]:
lr = 0.1

for epoch in range(1000):
  loss = 0
  for i, j in zip(x, y):
    pred = nn.forward(x)
    loss += mse(y, pred)
    grad = d_mse(y, pred)
    nn.backward(grad)
    nn.update(lr)
  if epoch % 100 == 0:
    print(f"For Epoch {epoch}, total loss is {loss}")


For Epoch 0, total loss is inf
For Epoch 100, total loss is inf
For Epoch 200, total loss is inf
For Epoch 300, total loss is inf
For Epoch 400, total loss is inf
For Epoch 500, total loss is inf
For Epoch 600, total loss is inf
For Epoch 700, total loss is inf
For Epoch 800, total loss is inf
For Epoch 900, total loss is inf
